In [3]:
import pandas as pd
import json

CSV_FILE = "bo_hyperparameter_sweep_results.csv"
OUTPUT_TABLE_FILE = "bo_summary_table.csv"

# Load results from CSV
df = pd.read_csv(CSV_FILE)

# Initialize lists for summary table
status_list = []
final_hv_list = []
cumulative_runtime_list = []

# Process each run in the CSV
for idx, row in df.iterrows():
    try:
        # Parse JSON strings
        hvs = json.loads(row["hypervolumes"])
        times = json.loads(row["times"])

        # Determine if run completed (length of HV array should be N_ITER + 1 if finished)
        if len(hvs) == 11:  # Assuming N_ITER = 10
            status = "Finished"
            final_hv = hvs[-1]
            cumulative_runtime = sum(times)
        else:
            status = "OOM Error"
            final_hv = hvs[-1] if hvs else None
            cumulative_runtime = sum(times) if times else None

    except Exception as e:
        status = "Error"
        final_hv = None
        cumulative_runtime = None

    status_list.append(status)
    final_hv_list.append(final_hv)
    cumulative_runtime_list.append(cumulative_runtime)

# Add new columns to the DataFrame
df["status"] = status_list
df["final_hv"] = final_hv_list
df["cumulative_runtime"] = cumulative_runtime_list

# Select and reorder columns for summary table
summary_columns = [
    "raw_samples", "num_restarts", "batch_limit", "max_iter",
    "status", "final_hv", "cumulative_runtime"
]
summary_df = df[summary_columns]

# Save summary table
summary_df.to_csv(OUTPUT_TABLE_FILE, index=False)
print(f"✅ Summary table saved to {OUTPUT_TABLE_FILE}")

# Display first few rows of the summary table
summary_df


✅ Summary table saved to bo_summary_table.csv


,raw_samples,num_restarts,batch_limit,max_iter,status,final_hv,cumulative_runtime
0,128,10,5,100,Finished,114.935580,601.538366
1,256,10,5,100,OOM Error,114.944632,497.779800
2,512,10,5,100,Finished,114.903933,717.125897
3,1024,10,5,100,Finished,128.498322,1762.384579
4,256,5,5,100,Finished,114.734036,423.422371
5,256,10,5,100,Finished,115.089051,927.229674
6,256,20,5,100,Finished,115.080329,1693.359406
7,256,50,5,100,OOM Error,114.891079,4191.996632
8,256,10,1,100,Finished,114.993034,1249.293256
9,256,10,5,100,Finished,123.353210,1053.434638
